# Transformation

In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_key,
    ci.customer_id,
    ci.customer_key AS customer_number,
    ci.firstname,
    ci.lastname,
    la.country,
    ci.marital_status,
    CASE
        WHEN ci.gender <> 'n/a' THEN ci.gender
        ELSE COALESCE(ca.gender, 'n/a')
    END AS gender,
    ca.birthdate AS birthdate,
    ci.created_date AS create_date
FROM silver.crm_customers ci
LEFT JOIN silver.erp_customers ca
    ON ci.customer_key = ca.customer_id
LEFT JOIN silver.erp_customer_location la
    ON ci.customer_key = la.customer_number
"""
df = spark.sql(query)

In [0]:
df.limit(10).display()

# Write gold table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")

## Sanity checks for gold table

In [0]:
%sql
SELECT * FROM workspace.gold.dim_customers LIMIT 10